# MiBici Guadalajara — Star Schema Modeling

Builds the analytics-ready star schema (Fact_Trips, Dim_Station, Dim_Usuario, Dim_Date, and a station-distance bridge table) from the cleaned data, written into a new `analytics` schema.

## 1. Setup: connect and load cleaned data

In [5]:
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import os

load_dotenv()

DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

trips = pd.read_sql("SELECT * FROM cleaned.trips", con=engine)
stations = pd.read_sql("SELECT * FROM cleaned.stations", con=engine)

print(trips.shape, stations.shape)

(4532032, 14) (484, 5)


## 2. Check for users with inconsistent birth years before building Dim_Usuario

`Dim_Usuario` needs exactly one row per user, but `Año_de_nacimiento` and `Genero` are repeated on every trip in the source data. If a user's birth year differs across their own trips, we need a resolution rule before collapsing trip-level data down to one row per user. Checking whether `Genero` is also inconsistent for the same users helps distinguish the likely cause; a corrected profile value over time, versus the same `Usuario_Id` being used by more than one person.

In [9]:
birth_year_per_user = trips.groupby("Usuario_Id")["Año_de_nacimiento"].nunique()
inconsistent_ids = birth_year_per_user[birth_year_per_user > 1].index

genero_per_user = trips[trips["Usuario_Id"].isin(inconsistent_ids)].groupby("Usuario_Id")["Genero"].nunique()

print("Users with inconsistent birth year:", len(inconsistent_ids))
print("Of those, how many also have inconsistent Genero:", (genero_per_user > 1).sum())

Users with inconsistent birth year: 52
Of those, how many also have inconsistent Genero: 0


## 2b. Resolution rule for Dim_Usuario

Since none of the 52 users with inconsistent birth years also show inconsistent Genero, this points to profile corrections over time. Accordingly, each user's most recent trip record is used as the source of truth for their attributes, not the most frequent value, since a correction should supersede earlier (likely incorrect) entries.

In [13]:
dim_usuario = (
    trips.sort_values("Inicio_del_viaje")
    .groupby("Usuario_Id")
    .last()[["Genero", "Año_de_nacimiento"]]
    .reset_index()
)

print(dim_usuario.shape)
dim_usuario.head()

(31555, 3)


,Usuario_Id,Genero,Año_de_nacimiento
0,102,M,1982.0
1,201,M,1978.0
2,237,M,1980.0
3,272,M,1980.0
4,275,M,1984.0


## 3. Build Dim_Station

One row per station, sourced directly from the cleaned stations table. Note: the actual station file only contains `id`, `name`, `latitude`, `longitude`, `dpcapacity`, no `tipo` column was present in the real data, despite being mentioned in the original data description.

In [16]:
dim_station = stations.rename(columns={"id": "Station_Id"}).copy()

print(dim_station.shape)
dim_station.head()

(484, 5)


,Station_Id,name,latitude,longitude,dpcapacity
0,2,(GDL-001) C. Epigmenio Glez./ Av. 16 de Sept.,20.666378,-103.348820,15
1,3,(GDL-002) C. Colonias / Av. Niños héroes,20.667228,-103.366000,15
2,4,(GDL-003) C. Vidrio / Av. Chapultepec,20.667690,-103.368252,19
3,5,(GDL-004) C. Ghilardi /C. Miraflores,20.691847,-103.362549,19
4,6,(GDL-005) C. San Diego /Calzada Independencia,20.681158,-103.339363,11


## 4. Build the station-distance bridge table

Computes the Haversine (straight-line) distance for every unique Origen_Id/Destino_Id pair that actually occurs in the trip data. Calculated once per unique pair rather than per trip (4.5M rows); far cheaper, and the result joins onto Fact_Trips via (Origen_Id, Destino_Id).

In [19]:
import numpy as np

def haversine_vectorized(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

unique_pairs = trips[["Origen_Id", "Destino_Id"]].drop_duplicates().reset_index(drop=True)
print("Unique station pairs actually used:", len(unique_pairs))

Unique station pairs actually used: 79051


Join each unique pair to its origin and destination coordinates, then apply the vectorized Haversine calculation across all pairs at once.

In [23]:
station_coords = dim_station[["Station_Id", "latitude", "longitude"]]

bridge = unique_pairs.merge(
    station_coords, left_on="Origen_Id", right_on="Station_Id"
).rename(columns={"latitude": "origen_lat", "longitude": "origen_lon"}).drop(columns="Station_Id")

bridge = bridge.merge(
    station_coords, left_on="Destino_Id", right_on="Station_Id"
).rename(columns={"latitude": "destino_lat", "longitude": "destino_lon"}).drop(columns="Station_Id")

bridge["distance_km"] = haversine_vectorized(
    bridge["origen_lat"], bridge["origen_lon"],
    bridge["destino_lat"], bridge["destino_lon"]
)

bridge = bridge[["Origen_Id", "Destino_Id", "distance_km"]]

print(bridge.shape)
bridge.describe()

(79051, 3)


,Origen_Id,Destino_Id,distance_km
count,79051.00000,79051.000000,79051.000000
mean,189.31968,189.967451,2.865209
std,117.75777,120.437550,1.591758
min,2.00000,2.000000,0.000000
25%,74.00000,70.000000,1.639284
50%,190.00000,191.000000,2.669784
75%,281.00000,284.000000,3.864953
max,402.00000,402.000000,12.583932


## 5. Write Dim_Usuario, Dim_Station, and the distance bridge to the analytics schema

In [26]:
with engine.connect() as conn:
    conn.execute(text("CREATE SCHEMA IF NOT EXISTS analytics;"))
    conn.commit()

dim_usuario.to_sql("dim_usuario", con=engine, schema="analytics", if_exists="replace", index=False)
dim_station.to_sql("dim_station", con=engine, schema="analytics", if_exists="replace", index=False)
bridge.to_sql("bridge_station_distance", con=engine, schema="analytics", if_exists="replace", index=False)

print("Dim_Usuario:", dim_usuario.shape[0], "rows")
print("Dim_Station:", dim_station.shape[0], "rows")
print("Bridge_Station_Distance:", bridge.shape[0], "rows")

Dim_Usuario: 31555 rows
Dim_Station: 484 rows
Bridge_Station_Distance: 79051 rows


## 6. Build Dim_Date

A standard calendar dimension spanning the full range of trip dates in 2025, with derived attributes (weekday, month name, is_weekend) precomputed. This is what enables clean time-based slicing and comparisons in Power BI without repeated DAX calculations.

In [29]:
date_range = pd.date_range(start=trips["Inicio_del_viaje"].dt.date.min(),
                             end=trips["Inicio_del_viaje"].dt.date.max(), freq="D")

dim_date = pd.DataFrame({"Date": date_range})
dim_date["Date_Id"] = dim_date["Date"].dt.strftime("%Y%m%d").astype(int)
dim_date["Year"] = dim_date["Date"].dt.year
dim_date["Month"] = dim_date["Date"].dt.month
dim_date["Month_Name"] = dim_date["Date"].dt.month_name()
dim_date["Day"] = dim_date["Date"].dt.day
dim_date["Weekday_Name"] = dim_date["Date"].dt.day_name()
dim_date["Is_Weekend"] = dim_date["Date"].dt.dayofweek.isin([5, 6])

print(dim_date.shape)
dim_date.head()

(365, 8)


,Date,Date_Id,Year,Month,Month_Name,Day,Weekday_Name,Is_Weekend
0,2025-01-01,20250101,2025,1,January,1,Wednesday,False
1,2025-01-02,20250102,2025,1,January,2,Thursday,False
2,2025-01-03,20250103,2025,1,January,3,Friday,False
3,2025-01-04,20250104,2025,1,January,4,Saturday,True
4,2025-01-05,20250105,2025,1,January,5,Sunday,True


## 7. Build Fact_Trips

The fact table: one row per trip, referencing dimensions via keys rather than duplicating descriptive attributes. `Duracion_min` and the quality flags carry over directly; `Año_de_nacimiento` and `Genero` are dropped here since they now live in `Dim_Usuario` and would be redundant (and, per the resolution rule, potentially inconsistent with a trip's own timestamp if repeated per-row).

In [32]:
fact_trips = trips.copy()

fact_trips["Date_Id"] = fact_trips["Inicio_del_viaje"].dt.strftime("%Y%m%d").astype(int)
fact_trips["Hora_Inicio"] = fact_trips["Inicio_del_viaje"].dt.hour

fact_trips = fact_trips[[
    "Viaje_Id", "Usuario_Id", "Origen_Id", "Destino_Id", "Date_Id",
    "Hora_Inicio", "Inicio_del_viaje", "Fin_del_viaje", "Duracion_min",
    "is_valid_duration", "is_valid_birth_year", "is_valid_genero",
    "origen_id_valid", "destino_id_valid"
]]

print(fact_trips.shape)
fact_trips.head()

(4532032, 14)


,Viaje_Id,Usuario_Id,Origen_Id,Destino_Id,Date_Id,Hora_Inicio,Inicio_del_viaje,Fin_del_viaje,Duracion_min,is_valid_duration,is_valid_birth_year,is_valid_genero,origen_id_valid,destino_id_valid
0,37162342,601273,211,395,20250101,0,2025-01-01 00:00:44,2025-01-01 00:11:29,10.750000,True,True,True,True,True
1,37162343,1571524,35,260,20250101,0,2025-01-01 00:04:11,2025-01-01 00:15:07,10.933333,True,True,True,True,True
2,37162344,1435200,35,260,20250101,0,2025-01-01 00:05:27,2025-01-01 00:15:23,9.933333,True,True,True,True,True
3,37162345,126327,190,11,20250101,0,2025-01-01 00:07:47,2025-01-01 00:17:42,9.916667,True,True,True,True,True
4,37162346,416121,273,8,20250101,0,2025-01-01 00:08:52,2025-01-01 00:18:10,9.300000,True,True,True,True,True


## 8. Write Fact_Trips and Dim_Date to the analytics schema

In [37]:
fact_trips.to_sql("fact_trips", con=engine, schema="analytics", if_exists="replace", index=False)
dim_date.to_sql("dim_date", con=engine, schema="analytics", if_exists="replace", index=False)

print("Fact_Trips:", fact_trips.shape[0], "rows")
print("Dim_Date:", dim_date.shape[0], "rows")

Fact_Trips: 4532032 rows
Dim_Date: 365 rows
